# Time-Matched WIND-L Run

This notebook trains the largest WIND model of the pilot (`wind_l`) for about as much wall-clock time as SpherIDiff needs in the pilot, instead of for the same number of steps. It is a separate run from the main pilot, and its results are stored separately. The main pilot compares the models at equal steps and samples, which favours the model that is slower per sample. This run tests whether the lead of SpherIDiff remains when WIND receives the extra time.

The run uses the same data, batch size, learning rate and evaluation as the pilot, and three times as many steps (135,000 against 45,000), with its own full cosine schedule. The step count is an estimate from the measured speeds, and it should be adjusted when the timing of the pilot is known. The notebook is designed for a background run, in which Kaggle executes the whole notebook and keeps the files of `/kaggle/working` when it ends. The results are packed into `pilot_timematched_results.zip` in the working folder.

After the training, the run is evaluated under the same fixed final protocol as the main pilot (`configs/protocol/final_v1.yaml`) on the held-out test year, so that its results can be compared directly with those of the main pilot.

In [ ]:
import os
import sys
import time
from pathlib import Path

import torch

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUNS_ROOT = WORK_ROOT / "runs"
FINAL_PROTOCOL = "final_v1"
RUN_NAME = "pilot_wind_l_long"
REFERENCE_RUNS = ["pilot_spheridiff_b4", "pilot_wind_l"]
ARCHIVE = WORK_ROOT / "pilot_timematched_results.zip"

## Inputs and hardware

The repository code is cloned from GitHub (`https://github.com/syeddaniyalg/spheri-diff`) into the working folder, which requires internet access for the notebook. The data cache is read from the attached dataset `thesyeddaniyal/era5-pilot-6degrees`, at `/kaggle/input/datasets/thesyeddaniyal/era5-pilot-6degrees`. The folder that holds `stats.npz` is used as the cache root. When that input path does not exist, the whole of `/kaggle/input` is searched for `stats.npz`.

The runs of the main pilot (`pilot_spheridiff_b4` and `pilot_wind_l`) are used as references when they are attached as a Kaggle input, for example as the output of the main pilot notebook. They are found by the folder name and the `evaluations.jsonl` file inside. Without them, the run is trained and stored, and the comparison is left out.

In [ ]:
import subprocess

REPOSITORY_URL = "https://github.com/syeddaniyalg/spheri-diff"
CODE_ROOT = WORK_ROOT / "spheri-diff"
DATASET_ROOT = INPUT_ROOT / "datasets" / "thesyeddaniyal" / "era5-pilot-6degrees"


def locate(marker, root):
    for current, _, files in os.walk(root):
        if marker in files:
            return Path(current)
    raise FileNotFoundError(f"{marker} was not found in {root}")


def locate_reference(name):
    for root in (INPUT_ROOT, WORK_ROOT):
        for current, _, files in os.walk(root):
            if Path(current).name == name and "evaluations.jsonl" in files:
                return Path(current).parent
    return None


if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)
CACHE_ROOT = locate("stats.npz", DATASET_ROOT if DATASET_ROOT.exists() else INPUT_ROOT)
REFERENCE_ROOT = locate_reference(REFERENCE_RUNS[0])
sys.path.insert(0, str(CODE_ROOT / "src"))
print("code:", CODE_ROOT)
print("data:", CACHE_ROOT)
print("reference runs:", REFERENCE_ROOT)
for index in range(torch.cuda.device_count()):
    print("gpu", index, torch.cuda.get_device_name(index))

## Running the job

The run is a single process with its own log file in `runs/_logs`, on the first GPU. The notebook prints every line that the job writes as soon as it appears, prefixed with the elapsed time, the GPU and the job name. The same lines are kept in `runs/_logs/parallel.log`.

The job and the packing of the results share one cell with a `try` and a `finally`. When the job fails, or the cell is interrupted, the figures, the status file and the archive are still produced. The archive is also refreshed every 30 minutes during the run. The job has no time limit of its own and runs until Kaggle ends the session. Checkpoints at 25%, 50% and 75% of the steps are kept. The final evaluation starts when all training jobs have ended. The final evaluation writes `final_evaluation.json` into each run folder, and the table `final_evaluation_summary.txt` lists all runs under the same protocol.

In [ ]:
from spheridiff.kaggle.finalise import finalise, refresh_archive
from spheridiff.kaggle.jobs import assign_queues, final_job, train_job
from spheridiff.kaggle.runner import ParallelRunner, RunContext

GPUS = torch.cuda.device_count()
LOGS = RUNS_ROOT / "_logs"
TRAINING_CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, LOGS, CACHE_ROOT)
EVALUATION_CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, LOGS, CACHE_ROOT)
COMPARISONS = [(RUN_NAME, name) for name in REFERENCE_RUNS] if REFERENCE_ROOT else []


def refresh():
    return refresh_archive(RUNS_ROOT, ARCHIVE)


RUNNER = None
EVALUATOR = None
try:
    RUNNER = ParallelRunner(assign_queues([[train_job(RUN_NAME)]], GPUS), TRAINING_CONTEXT, refresh=refresh)
    RUNNER.run()
    EVALUATION_QUEUES = assign_queues([[final_job(RUN_NAME, FINAL_PROTOCOL)]], GPUS)
    EVALUATOR = ParallelRunner(EVALUATION_QUEUES, EVALUATION_CONTEXT, refresh=refresh)
    EVALUATOR.run()
finally:
    OUTCOMES = [*(RUNNER.outcomes() if RUNNER else []), *(EVALUATOR.outcomes() if EVALUATOR else [])]
    finalise(
        RUNS_ROOT,
        ARCHIVE,
        OUTCOMES,
        [RUN_NAME],
        comparisons=COMPARISONS,
        reference_root=REFERENCE_ROOT,
        at_final=True,
        final_names=[RUN_NAME],
        final_reference_names=REFERENCE_RUNS if REFERENCE_ROOT else [],
    )
print(OUTCOMES)
print(ARCHIVE, f"{ARCHIVE.stat().st_size / 1024**2:.1f} MB")

## Results

The summary lists the steps, training time, speed and final loss of the run. When the reference runs are available, the comparison summary lists, for the time-matched run against each reference, the step at which it reaches the final error of the reference and stays there, the errors at fixed fractions of training, the mean error over the whole curve, and the paired difference of the final errors with a 95% interval over the evaluation windows. The figures show the time-matched run together with the references, against steps, samples seen and training time. The final evaluation table lists, for each run, the number of parameters, the trained steps, the forecast error and CRPS, and for each task and method the normalised error on the unobserved entries, with the reference of the task as the last row.

In [ ]:
import json

summary = RUNS_ROOT / RUN_NAME / "summary.json"
print(RUN_NAME, json.loads(summary.read_text()) if summary.exists() else "no summary, see runs/_logs")

comparison = RUNS_ROOT / "comparison_summary.txt"
if comparison.exists():
    print(comparison.read_text(encoding="utf-8"))

final_summary = RUNS_ROOT / "final_evaluation_summary.txt"
if final_summary.exists():
    print(final_summary.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import Image, display

for figure in ["evaluation_curves.png", "training_loss.png"]:
    path = RUNS_ROOT / figure
    if path.exists():
        display(Image(filename=str(path)))